# 단어의 표현 (Word Representation)


기계는 문자를 그대로 인식할 수 없기때문에 숫자로 변환



# 1 원-핫 인코딩 (One-Hot Encoding)

## 1.1 직접 구현해보기

### "원숭이, 바나나, 사과" 로 원-핫 인코딩을 한다면

In [1]:
# 인코딩 대상 단어들을 담은 리스트
word_ls = ['원숭이','바나나','사과']

In [2]:
# 원핫 인코딩 생성
def one_hot_encode(words):
    unique_words = list(set(words))
    # unique_words = ['원숭이','바나나','사과']

    # list.index() 순서 -> 해당 위치가 1
    # 원숭이 -> 0번, [1, 0, 0] -> result에 추가
    # 바나나 -> 1번, [0, 1, 0] -> result
    result = []
    for word in words:
        idx = unique_words.index(word)
        temp = [0] * len(unique_words)
        temp[idx] = 1
        result.append(temp)

    return result
# [
#     [1, 0, 0],
#     [0, 1, 0],
#     [0, 0, 1]
# ]

In [5]:
one_hot_vectors = one_hot_encode(word_ls)
one_hot_vectors

[[0, 0, 1], [0, 1, 0], [1, 0, 0]]

### "코끼리"라는 단어가 추가된다면?

In [6]:
word_ls = ['원숭이','바나나','사과','코끼리']

In [8]:
one_hot_vectors = one_hot_encode(word_ls + ['바나나'])
one_hot_vectors

[[0, 0, 0, 1], [0, 1, 0, 0], [1, 0, 0, 0], [0, 0, 1, 0], [0, 1, 0, 0]]

In [ ]:
# 라벨인코딩 ? word2idx => {}
# 원숭이: 0
# 바나나: 1
# 사과: 2
# 코끼리: 3

[0, 1, 2, 3, 1]

## 1.3 sklearn 활용

- LabelEncoder() : 유니크한 범주형 값들을 1부터 증가하는 숫자로 인코딩
- OneHotEncoder() : 유니크한 범주형 값들을 n:1 행렬로 변환

함수명 | 설명
--|--
fit(X[, y])	| Fit OneHotEncoder to X.
fit_transform(X[, y])	| Fit OneHotEncoder to X, then transform X.
inverse_transform(X)	| Convert the back data to the original representation.
transform(X)	| Transform X using one-hot encoding.

In [ ]:
%pip install scikit-learn

In [12]:
# sklearn을 활용한 one-hot encoding
from numpy import array
from numpy import argmax
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder

# 예제 데이터 배열
values = array(word_ls)
print(values)

# 문자열에 숫자를 붙임
label_enc = LabelEncoder()
int_enc = label_enc.fit_transform(values)
print(int_enc)

# binary encode
onehot_enc = OneHotEncoder()
int_enc = int_enc.reshape(len(int_enc), 1) # n:1 matrix로 변환
print(int_enc)
onehot_enc = onehot_enc.fit_transform(int_enc)
print(onehot_enc.toarray())

# one-hot encoding 의 첫번째 배열을 값을 역으로 산출
inverted = label_enc.inverse_transform([argmax(onehot_enc[0, :])])
print(inverted)

['원숭이' '바나나' '사과' '코끼리']
[2 0 1 3]
[[2]
 [0]
 [1]
 [3]]
[[0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 0. 1.]]
['원숭이']




---



# 2 밀집 벡터 (Dense Vector)

## 2-1 유사도 계산

### 2.1.1 유클리디언 거리(Euclidean distance)
두 벡터사이의 직선 거리. 피타고라스 정리를 생각하면 이해하기 쉬움

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/55/Euclidean_distance_2d.svg/220px-Euclidean_distance_2d.svg.png"  width="200"/>

<img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/795b967db2917cdde7c2da2d1ee327eb673276c0" width="350"/>

https://en.wikipedia.org/wiki/Euclidean_distance

In [18]:
word_embedding_dic = {
    '사과' : [1.0, 0.5],
    '바나나' : [0.9, 1.2],
    '원숭이' : [0.5, 1.5]
}

In [29]:
import numpy as np
# 유클리디언 거리 계산

def euclidean_dist(vec1, vec2):
    # assert len(vec1)==len(vec2), '벡터 차원 크기가 다릅니다.'
    # python 방식
    dist = 0.0
    for i in range(len(vec1)):
        dist += (vec1[i] - vec2[i]) ** 2
    dist = dist ** (1/2)

    # numpy 방식
    vec1, vec2 = np.array(vec1), np.array(vec2)
    dist = ((vec1-vec2)**2).sum() ** (1/2)
    return dist

# 사과와 바나나의 코사인 유사도
euclidean_dist(word_embedding_dic['사과'], word_embedding_dic['바나나'])

0.7071067811865475

### 2.1.2 자카드 유사도(Jaccard index)

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/1f/Intersection_of_sets_A_and_B.svg/200px-Intersection_of_sets_A_and_B.svg.png" />

<img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/eaef5aa86949f49e7dc6b9c8c3dd8b233332c9e7" />

https://en.wikipedia.org/wiki/Jaccard_index

In [31]:
s1 = '대부분 원숭이는 바나나를 좋아합니다.'
s2 = '코주부 원숭이는 바나나를 싫어합니다.'

# 토큰화를 수행합니다.
token_s1 = s1.split()
token_s2 = s2.split()

# 자카드 유사도 계산
set_s1 = set(token_s1)
set_s2 = set(token_s2)

jaccard_sim = len(set_s1 & set_s2) / len(set_s1 | set_s2)
print(jaccard_sim)

0.3333333333333333


### 2.1.3 코사인 유사도(Cosine Similarity)

*  두 벡터간의 유사도를 측정하는 방법 중 하나
*  두 벡터 사이의 코사인을 측정
*  0도 = 1, 90도 = 0, 180도 = -1   ==> 1에 가까울수록 유사도가 높음




<img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/1d94e5903f7936d3c131e040ef2c51b473dd071d" width='350'/>

https://en.wikipedia.org/wiki/Cosine_similarity

In [ ]:
# 코사인 유사도 계산

# np.linalg.norm()

def cosine_similarity(vec1, vec2):
    cos_sim = 0.0
    # python 방식
    vec1_size = 0.0
    vec2_size = 0.0
    for i in range(len(vec1)):
        cos_sim += vec1[i] * vec2[i]
        vec1_size += vec1[i]**2
        vec2_size += vec2[i]**2
    cos_sim = cos_sim / (vec1_size * vec2_size) ** (1/2)

    # numpy 활용
    vec1, vec2 = np.array(vec1), np.array(vec2)
    cos_sim = (vec1 * vec2).sum() / np.sqrt( (vec1**2).sum() * (vec2**2).sum() )
    cos_sim = (vec1 * vec2).sum() / ( np.linalg.norm(vec1) * np.linalg.norm(vec2) )

    return cos_sim

In [63]:
# 사과와 바나나의 코사인 유사도
print(cosine_similarity(word_embedding_dic['사과'], word_embedding_dic['바나나']))
print(euclidean_dist(word_embedding_dic['사과'], word_embedding_dic['바나나']))

0.8944271909999159
0.7071067811865475


In [64]:
# 사과와 원숭이의 코사인 유사도
print(cosine_similarity(word_embedding_dic['사과'], word_embedding_dic['원숭이']))
print(euclidean_dist(word_embedding_dic['사과'], word_embedding_dic['원숭이']))

0.7071067811865475
1.118033988749895


In [65]:
    # 바나나와 원숭이의 코사인 유사도
print(cosine_similarity(word_embedding_dic['바나나'], word_embedding_dic['원숭이']))
print(euclidean_dist(word_embedding_dic['바나나'], word_embedding_dic['원숭이']))

0.9486832980505138
0.5




---



# 3 TF-IDF를 활용한 단어 벡터

### 3.1 직접 구현하기

weighting schema|weight|설명
--|--|--
term frequency|<img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/91699003abf4fe8bdf861bbce08e73e71acf5fd4" />|=토큰빈도/문서내토큰빈도
inverse document frequency|<img src="https://wikimedia.org/api/rest_v1/media/math/render/svg/864fcfdc0c16344c11509f724f1aa7081cf9f657" />|=log(총문서갯수/(토큰이 등장한 문서수))

In [66]:
d1 = "The cat sat on my face I hate a cat"
d2 = "The dog sat on my bed I love a dog"

In [67]:
corpus = [d1, d2]

In [81]:
import numpy as np

# TF-IDF 계산

# 0-0. 토큰화
doc_tokens = []
unique_tokens = set()
for doc in corpus:
    doc_tokens.append(doc.split())
    unique_tokens.update(doc.split())
# print(unique_tokens)

# 0-1. word2idx 생성
word2idx = {t: i for i, t in enumerate(unique_tokens)}
# print(word2idx)

# document 내 token 등장비율 TF
tfs = []
for tokens in doc_tokens:
    tf = []
    total = len(tokens)
    for token, idx in word2idx.items():
        tf.append( tokens.count(token) / total )
    tfs.append(tf)

print('TF')
print(*tfs, sep='\n')

# 단어 - 문서 등장 횟수 IDF
idfs = []
for token, idx in word2idx.items():
    df = 0
    total = len(doc_tokens)
    for tokens in doc_tokens:
        df += 1 if token in tokens else 0
    idfs.append( np.log10(total / df) )

print('-'*64 + '\n')
print('IDF')
print(idfs)

# 두 값의 곱
tfidfs = []
for tf in tfs:
    tfidf = [t * idf for t, idf in zip(tf, idfs)]
    tfidfs.append(tfidf)

print('-'*64 + '\n')
print('TF-IDF')
print(*tfidfs, sep='\n')

TF
[0.2, 0.1, 0.0, 0.1, 0.1, 0.1, 0.1, 0.1, 0.0, 0.0, 0.1, 0.1]
[0.0, 0.0, 0.1, 0.1, 0.0, 0.1, 0.1, 0.1, 0.1, 0.2, 0.1, 0.1]
----------------------------------------------------------------

IDF
[np.float64(0.3010299956639812), np.float64(0.3010299956639812), np.float64(0.3010299956639812), np.float64(0.0), np.float64(0.3010299956639812), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.3010299956639812), np.float64(0.3010299956639812), np.float64(0.0), np.float64(0.0)]
----------------------------------------------------------------

TF-IDF
[np.float64(0.06020599913279624), np.float64(0.03010299956639812), np.float64(0.0), np.float64(0.0), np.float64(0.03010299956639812), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
[np.float64(0.0), np.float64(0.0), np.float64(0.03010299956639812), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.03010299956639812)

In [114]:
# numpy 활용
# 0-0. 토큰화
doc_tokens = []
unique_tokens = set()
for doc in corpus:
    doc_tokens.append([t for t in doc.split() if t not in ['I', 'a']])
    unique_tokens.update(doc.split())

# 0-1. word2idx 생성
word2idx = {t: i for i, t in enumerate(unique_tokens) if t not in ['I', 'a']}
print(word2idx)
print(*doc_tokens, sep='\n')

# TDM
tdm = np.zeros((len(word2idx), len(doc_tokens)))

for d_i, tokens in enumerate(doc_tokens):
    for token, t_i in word2idx.items():
        tdm[t_i, d_i] = tokens.count(token)
# print(tdm)

# tf
tf = tdm / tdm.sum(axis=0)
print(tf)

# idf
idf = np.log10( len(doc_tokens) / (tdm > 0).sum(axis=1) )
print(idf)

# tf-idf
tfidf = tf * idf.reshape(-1, 1)
tfidf

{'cat': 0, 'hate': 1, 'bed': 2, 'The': 3, 'face': 4, 'sat': 5, 'on': 6, 'my': 7, 'love': 8, 'dog': 9}
['The', 'cat', 'sat', 'on', 'my', 'face', 'hate', 'cat']
['The', 'dog', 'sat', 'on', 'my', 'bed', 'love', 'dog']
[[0.25  0.   ]
 [0.125 0.   ]
 [0.    0.125]
 [0.125 0.125]
 [0.125 0.   ]
 [0.125 0.125]
 [0.125 0.125]
 [0.125 0.125]
 [0.    0.125]
 [0.    0.25 ]]
[0.30103 0.30103 0.30103 0.      0.30103 0.      0.      0.      0.30103
 0.30103]


array([[0.0752575 , 0.        ],
       [0.03762875, 0.        ],
       [0.        , 0.03762875],
       [0.        , 0.        ],
       [0.03762875, 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.        ],
       [0.        , 0.03762875],
       [0.        , 0.0752575 ]])

In [102]:
# %pip install pandas

In [101]:
import pandas as pd
df = pd.DataFrame(tfidf, index=word2idx, columns=['doc1', 'doc2'])
df.round(4)

,doc1,doc2
cat,0.0602,0.0000
hate,0.0301,0.0000
bed,0.0000,0.0301
The,0.0000,0.0000
face,0.0301,0.0000
sat,0.0000,0.0000
on,0.0000,0.0000
my,0.0000,0.0000
love,0.0000,0.0301
dog,0.0000,0.0602


### 3.2 sklearn 활용

- feature_extraction.text.CountVectorizer() : 문장들에서 단어의 빈도수를 벡터화

In [103]:
from sklearn.feature_extraction.text import CountVectorizer


d1 = "The cat sat on my face I hate a cat"
d2 = "The dog sat on my bed I love a dog"
corpus = [d1, d2]
count_vect = CountVectorizer()
countv = count_vect.fit_transform(corpus)
print(countv.toarray()) # 코퍼스로부터 각 단어의 빈도 수를 기록한다.
print(count_vect.vocabulary_) # 각 단어의 인덱스가 어떻게 부여되었는지를 보여준다.

[[0 2 0 1 1 0 1 1 1 1]
 [1 0 2 0 0 1 1 1 1 1]]
{'the': 9, 'cat': 1, 'sat': 8, 'on': 7, 'my': 6, 'face': 3, 'hate': 4, 'dog': 2, 'bed': 0, 'love': 5}


- feature_extraction.text.TfidfVectorizer() : 문장들을 TF-IDF로 벡터화

In [104]:
from sklearn.feature_extraction.text import TfidfVectorizer

d1 = "The cat sat on my face I hate a cat"
d2 = "The dog sat on my bed I love a dog"
corpus = [d1, d2]
tfidf_vect = TfidfVectorizer().fit(corpus)
tfidfv = tfidf_vect.transform(corpus)
print(tfidfv.toarray())
print(tfidf_vect.vocabulary_)

[[0.         0.70600557 0.         0.35300279 0.35300279 0.
  0.25116439 0.25116439 0.25116439 0.25116439]
 [0.35300279 0.         0.70600557 0.         0.         0.35300279
  0.25116439 0.25116439 0.25116439 0.25116439]]
{'the': 9, 'cat': 1, 'sat': 8, 'on': 7, 'my': 6, 'face': 3, 'hate': 4, 'dog': 2, 'bed': 0, 'love': 5}


## 3.3 gensim 활용

- corpora.Dictionary - 토큰화한 문서들 내에서 index-단어 사전을 생성
- models.TfidfModel() - TF-IDF 벡터화

In [ ]:
%pip install gensim

In [116]:
import gensim.downloader as api
from gensim.models import TfidfModel
from gensim import corpora

d1 = "The cat sat on my face I hate a cat"
d2 = "The dog sat on my bed I love a dog"
corpus = [d1, d2]

doc_ls = [doc.split() for doc in corpus]
id2word = corpora.Dictionary(doc_ls)  # fit dictionary
corpus = [id2word.doc2bow(doc) for doc in doc_ls]  # convert corpus to BoW format

tfidf = TfidfModel(corpus)  # fit model
vector = tfidf[corpus[0]]  # apply model to the first corpus document

In [117]:
tfidf[corpus][0]

[(3, np.float64(0.8164965809277261)),
 (4, np.float64(0.4082482904638631)),
 (5, np.float64(0.4082482904638631))]



---



# 4 LSA(Latent Semantic Analysis)를 활용한 단어 벡터

## 4.1 sklearn 활용

- decomposition.TruncatedSVD() : 특이값분해

In [118]:
doc_ls = [
    '바나나 사과 포도 포도',
    '사과 포도',
    '포도 바나나',
    '짜장면 짬뽕 탕수육',
    '볶음밥 탕수육',
    '짜장면 짬뽕',
    '라면 스시',
    '스시',
    '가츠동 스시 소바',
    '된장찌개 김치찌개 김치',
    '김치 된장',
    '비빔밥 김치'
]

In [119]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import TruncatedSVD

count_vect = CountVectorizer()
countv = count_vect.fit_transform(doc_ls) # TDM -> token 개수, doc 개수 
print(countv)
svd = TruncatedSVD(n_components=2, algorithm='randomized', n_iter=100)
svd.fit(countv)

features = count_vect.get_feature_names_out() # 단어 집합. 1,000개의 단어가 저장됨.
for i in range(len(features)) :
    print("{} : {}".format(features[i], svd.components_[:,i]))

  (np.int32(0), np.int32(6))	1
  (np.int32(0), np.int32(9))	1
  (np.int32(0), np.int32(15))	2
  (np.int32(1), np.int32(9))	1
  (np.int32(1), np.int32(15))	1
  (np.int32(2), np.int32(6))	1
  (np.int32(2), np.int32(15))	1
  (np.int32(3), np.int32(12))	1
  (np.int32(3), np.int32(13))	1
  (np.int32(3), np.int32(14))	1
  (np.int32(4), np.int32(14))	1
  (np.int32(4), np.int32(7))	1
  (np.int32(5), np.int32(12))	1
  (np.int32(5), np.int32(13))	1
  (np.int32(6), np.int32(5))	1
  (np.int32(6), np.int32(11))	1
  (np.int32(7), np.int32(11))	1
  (np.int32(8), np.int32(11))	1
  (np.int32(8), np.int32(0))	1
  (np.int32(8), np.int32(10))	1
  (np.int32(9), np.int32(4))	1
  (np.int32(9), np.int32(2))	1
  (np.int32(9), np.int32(1))	1
  (np.int32(10), np.int32(1))	1
  (np.int32(10), np.int32(3))	1
  (np.int32(11), np.int32(1))	1
  (np.int32(11), np.int32(8))	1
가츠동 : [ 1.58850124e-18 -8.32576020e-17]
김치 : [9.43945330e-19 5.18274674e-15]
김치찌개 : [3.90125549e-19 2.16962965e-15]
된장 : [1.51090239e-19 1.4393538

## 4.2 gensim 활용

- models.LsiModel() : LSA 모델

In [120]:
doc_ls = [
    '바나나 사과 포도 포도',
    '사과 포도',
    '포도 바나나',
    '짜장면 짬뽕 탕수육',
    '볶음밥 탕수육',
    '짜장면 짬뽕',
    '라면 스시',
    '스시',
    '가츠동 스시 소바',
    '된장찌개 김치찌개 김치',
    '김치 된장',
    '비빔밥 김치'
]
doc_ls = [d.split() for d in doc_ls]

In [121]:
from gensim import corpora
from gensim.models import LsiModel

id2word = corpora.Dictionary(doc_ls) #사전 구축
corpus = [id2word.doc2bow(text) for text in doc_ls] # 코퍼스 생성
lsi = LsiModel(corpus, id2word=id2word, num_topics=2) #LSA 모델

for i in id2word.keys() :
    print("{} : {}".format(id2word[i], lsi.projection.u[i]))

바나나 : [4.08248290e-01 2.07057026e-16]
사과 : [4.08248290e-01 1.67413166e-16]
포도 : [ 8.16496581e-01 -1.03944364e-16]
짜장면 : [-6.23621443e-17 -6.12024764e-01]
짬뽕 : [-1.29026680e-16 -6.12024764e-01]
탕수육 : [-3.46940358e-17 -4.84287758e-01]
볶음밥 : [ 1.09307172e-17 -1.27737006e-01]
라면 : [ 1.41641337e-17 -5.86275194e-16]
스시 : [ 8.80951179e-17 -1.90133250e-15]
가츠동 : [ 5.15979204e-18 -9.47221709e-16]
소바 : [ 3.32706148e-18 -9.83620944e-16]
김치 : [ 2.89901588e-16 -1.14807342e-14]
김치찌개 : [ 1.03563445e-16 -4.86241345e-15]
된장찌개 : [ 9.76412027e-17 -4.88756349e-15]
된장 : [ 4.23024927e-17 -3.32879993e-15]
비빔밥 : [ 1.30769913e-16 -3.31891459e-15]
